# Limpieza profesional con criterio

La limpieza mecánica aplica funciones sin pensar. La limpieza profesional toma decisiones justificadas.

Cada operación de limpieza tiene un impacto en el análisis posterior, y ese impacto debe ser evaluado antes de ejecutar la función.

La pregunta que guía cada decisión no es **¿qué función uso?** sino **¿qué impacto tiene esta decisión sobre la representatividad del dataset?**

## 1 — Tipos correctos antes de todo

Los tipos incorrectos producen errores silenciosos:

- Un string `"NaN"` no es detectado por `isnull()`
- Una columna de precios como string no se puede promediar
- Fechas como texto no se pueden ordenar cronológicamente

El primer paso siempre es verificar y corregir tipos antes de analizar nada.

In [2]:
import pandas as pd

df = pd.DataFrame({
    "id_cliente": [1, 2, 3, 4, 5],
    "nombre":     ["Ana", "Carlos", "Marta", None, "Pedro"],
    "edad":       ["32", "28", "none", "41", "36"],
    "importe":    [899.0, None, 349.0, 45.0, 25.0],
    "fecha":      ["2024-01-15", "2024-02-20", "2024-03-01", "15/03/2024", "2024-04-10"],
    "ciudad":     ["Madrid", "Barcelona", "madrid", "Sevilla", "Valencia"]
})

# "none" como string no es nulo para pandas — hay que convertirlo primero
df["edad"] = df["edad"].replace("none", pd.NA)
df["edad"] = pd.to_numeric(df["edad"])

# format='mixed' acepta mezcla de formatos en la misma columna
# dayfirst=True indica que el día va antes que el mes (15/03/2024)
df["fecha"] = pd.to_datetime(df["fecha"], format="mixed", dayfirst=True)

print(df.dtypes)

id_cliente             int64
nombre                   str
edad                 float64
importe              float64
fecha         datetime64[us]
ciudad                   str
dtype: object


## 2 — Nulos: detectar primero, decidir después

No todos los nulos se tratan igual. La decisión depende de qué representa el nulo y qué papel juega esa columna en el análisis.

In [ ]:
print(df.isnull().sum())

id_cliente    0
nombre        1
edad          1
importe       1
fecha         0
ciudad        0
dtype: int64


: 

### nombre nulo → `fillna`

El cliente existe (tiene `id_cliente`, `importe`, `fecha`) pero no tiene nombre registrado.

Eliminar la fila haría perder sus transacciones. Rellenar con `'Desconocido'` preserva el registro sin inventar datos.

In [ ]:
df["nombre"] = df["nombre"].fillna("Desconocido")
print(df['nombre'])

### edad nula → `fillna` con mediana

Variable numérica usada para análisis. Se rellena con la **mediana** en vez de la media porque la mediana no se distorsiona por outliers.

Ejemplo: si hay un cliente de 90 años, la media sube; la mediana no se mueve.

In [ ]:
mediana_edad = df["edad"].median()
df["edad"] = df["edad"].fillna(mediana_edad).astype(int)
print(f"Mediana usada para relleno: {mediana_edad}")
print(df['edad'])

### importe nulo → `dropna`

Dato crítico: un importe desconocido no se puede inventar. Rellenar con la media sería **fabricar una transacción**.

Se elimina la fila porque el análisis de revenue no puede incluir ventas sin importe real.

In [ ]:
print(f"Filas antes:  {len(df)}")
df = df.dropna(subset=["importe"])
print(f"Filas después: {len(df)}")

## 3 — Columnas con muchos nulos: cuándo eliminar la columna

Si una columna tiene más del 50–60 % de nulos, su capacidad informativa es tan baja que cualquier relleno sería más invención que dato.

Eliminar la columna es más honesto que conservarla rellena de valores fabricados.

El umbral va en una variable al inicio, no hardcodeado dentro del código.

In [ ]:
umbral_nulos = 0.5  # variable de configuración al inicio, no hardcodeada

columnas_con_demasiados_nulos = [
    col for col in df.columns
    if df[col].isnull().mean() > umbral_nulos
]

print(f"Columnas a eliminar: {columnas_con_demasiados_nulos}")
df = df.drop(columns=columnas_con_demasiados_nulos)

## 4 — Inconsistencias de formato

Pandas distingue entre `'Madrid'` y `'madrid'`: son dos valores distintos aunque semánticamente sean lo mismo.

Esto genera **grupos fantasma** en cualquier `groupby` o `value_counts`. `.str.title()` unifica la capitalización.

In [ ]:
print("Antes:")
print(df["ciudad"].value_counts())

df["ciudad"] = df["ciudad"].str.strip().str.title()

print("\nDespués:")
print(df["ciudad"].value_counts())

## 5 — Documentar cada decisión

En código profesional cada decisión de limpieza está documentada. No con comentarios genéricos sino con **justificaciones específicas**.

Esto permite que seis meses después (o cualquier otro analista) entienda por qué los datos tienen la forma que tienen.

In [ ]:
# DECISIONES DE LIMPIEZA — 2024-06-22

# nombre: rellenado con 'Desconocido' porque el cliente tiene
# transacciones asociadas y eliminar la fila haría perder revenue real

# edad: rellenada con mediana (34) en vez de media porque
# la distribución puede tener outliers en edades extremas

# importe: fila eliminada porque no se puede imputar el valor
# de una transacción financiera sin dato real

# ciudad: normalizada con str.title() para evitar grupos
# duplicados en groupby (madrid != Madrid para pandas)

print(df)

## 6 — Función reutilizable de limpieza

Cuando el mismo pipeline de limpieza se va a aplicar a varios datasets, tiene sentido encapsularlo en una función con parámetros configurables.

- `umbral_nulos`: porcentaje a partir del cual se elimina una columna entera
- `columnas_criticas`: columnas donde un nulo implica eliminar la fila completa

Una función por responsabilidad. Prints de control en cada operación.

In [ ]:
def limpiar_dataset(df, umbral_nulos=0.5, columnas_criticas=None):
    df = df.copy()

    # Eliminar columnas con demasiados nulos
    cols_a_eliminar = [c for c in df.columns if df[c].isnull().mean() > umbral_nulos]
    df = df.drop(columns=cols_a_eliminar)
    print(f'Columnas eliminadas por nulos: {cols_a_eliminar}')

    # Eliminar filas con nulos en columnas críticas
    if columnas_criticas:
        antes = len(df)
        df = df.dropna(subset=columnas_criticas)
        print(f'Filas eliminadas por nulos críticos: {antes - len(df)}')

    # Eliminar duplicados
    antes = len(df)
    df = df.drop_duplicates()
    print(f'Duplicados eliminados: {antes - len(df)}')

    return df


# Reconstruir el DataFrame original para demostrar la función desde cero
df_raw = pd.DataFrame({
    "id_cliente": [1, 2, 3, 4, 5],
    "nombre":     ["Ana", "Carlos", "Marta", None, "Pedro"],
    "edad":       ["32", "28", "none", "41", "36"],
    "importe":    [899.0, None, 349.0, 45.0, 25.0],
    "fecha":      ["2024-01-15", "2024-02-20", "2024-03-01", "15/03/2024", "2024-04-10"],
    "ciudad":     ["Madrid", "Barcelona", "madrid", "Sevilla", "Valencia"]
})

df_limpio = limpiar_dataset(df_raw, umbral_nulos=0.5, columnas_criticas=["importe"])
print(df_limpio)